<a href="https://colab.research.google.com/github/srividya76/srividya76/blob/main/Clinician_Record_Audio_and_then_convert_speech_to_text.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ffmpeg-python


In [ ]:
"""
Here are some of the possible references that helped with this code:
https://blog.addpipe.com/recording-audio-in-the-browser-using-pure-html5-and-minimal-javascript/
https://stackoverflow.com/a/18650249
https://hacks.mozilla.org/2014/06/easy-audio-capture-with-the-mediarecorder-api/
https://air.ghost.io/recording-to-an-audio-file-using-html5-and-js/
https://stackoverflow.com/a/49019356
"""
from IPython.display import HTML, Audio
from google.colab.output import eval_js
from base64 import b64decode
import numpy as np
from scipy.io.wavfile import read as wav_read
import io
import ffmpeg

AUDIO_HTML = """
<script>
var my_div = document.createElement("DIV");
var my_p = document.createElement("P");
var my_btn = document.createElement("BUTTON");
var t = document.createTextNode("Press to start recording");
my_btn.appendChild(t);
//my_p.appendChild(my_btn);
my_div.appendChild(my_btn);
document.body.appendChild(my_div);
var base64data = 0;
var reader;
var recorder, gumStream;
var recordButton = my_btn;
var handleSuccess = function(stream) {
  gumStream = stream;
  var options = {
    //bitsPerSecond: 8000, //chrome seems to ignore, always 48k
    mimeType : 'audio/webm;codecs=opus'
    //mimeType : 'audio/webm;codecs=pcm'
  };
  //recorder = new MediaRecorder(stream, options);
  recorder = new MediaRecorder(stream);
  recorder.ondataavailable = function(e) {
    var url = URL.createObjectURL(e.data);
    var preview = document.createElement('audio');
    preview.controls = true;
    preview.src = url;
    document.body.appendChild(preview);
    reader = new FileReader();
    reader.readAsDataURL(e.data);
    reader.onloadend = function() {
      base64data = reader.result;
      //console.log("Inside FileReader:" + base64data);
    }
  };
  recorder.start();
  };
recordButton.innerText = "Recording... press to stop";
navigator.mediaDevices.getUserMedia({audio: true}).then(handleSuccess);
function toggleRecording() {
  if (recorder && recorder.state == "recording") {
      recorder.stop();
      gumStream.getAudioTracks()[0].stop();
      recordButton.innerText = "Saving the recording... pls wait!"
  }
}
// https://stackoverflow.com/a/951057
function sleep(ms) {
  return new Promise(resolve => setTimeout(resolve, ms));
}
var data = new Promise(resolve=>{
//recordButton.addEventListener("click", toggleRecording);
recordButton.onclick = ()=>{
toggleRecording()
sleep(2000).then(() => {
  // wait 2000ms for the data to be available...
  // ideally this should use something like await...
  //console.log("Inside data:" + base64data)
  resolve(base64data.toString())
});
}
});

</script>
"""

def get_audio():
  display(HTML(AUDIO_HTML))
  data = eval_js("data")
  binary = b64decode(data.split(',')[1])

  process = (ffmpeg
    .input('pipe:0')
    .output('pipe:1', format='wav')
    .run_async(pipe_stdin=True, pipe_stdout=True, pipe_stderr=True, quiet=True, overwrite_output=True)
  )
  output, err = process.communicate(input=binary)

  riff_chunk_size = len(output) - 8
  # Break up the chunk size into four bytes, held in b.
  q = riff_chunk_size
  b = []
  for i in range(4):
      q, r = divmod(q, 256)
      b.append(r)

  # Replace bytes 4:8 in proc.stdout with the actual size of the RIFF chunk.
  riff = output[:4] + bytes(b) + output[8:]

  sr, audio = wav_read(io.BytesIO(riff))
  return audio, sr


In [ ]:
audio, sr = get_audio()


In [ ]:
import scipy
scipy.io.wavfile.write('/content/sample_data/audio_recording.wav', sr, audio)

In [ ]:
!apt-get install -y portaudio19-dev
!pip install sounddevice

In [ ]:
# Installing Whisper libary
!pip install git+https://github.com/openai/whisper.git -q
import whisper

In [ ]:
model1 = whisper.load_model('small')
text = model1.transcribe('/content/sample_data/audio_recording.wav')
#printing the transcribe
text['text']

In [ ]:
!pip install openai
import openai

# Set up OpenAI API credentials
openai.api_key = 'sk-'

def extract_medical_info(transcript):
    # Specify the prompt for the model
    prompt = f"Extract medical and health information from the following transcript:\n\n{transcript}\n\n---\n\nMedical information:"

    # Generate a response using the language model
    response = openai.Completion.create(
        engine='text-davinci-003',  # Use the GPT-3.5 model
        prompt=prompt,
        max_tokens=200,  # Adjust as needed
        n=1,  # Generate a single response
        stop=None,  # Allow the model to generate a complete response
        temperature=0.5,  # Adjust as needed
        top_p=1.0,
        frequency_penalty=0.0,
        presence_penalty=0.0
    )

    # Extract the generated text from the response
    generated_text = response.choices[0].text.strip()

    # Return the extracted medical information
    return generated_text

# Example usage
transcript = """
How are you doing ? How is your son? How's everyone at home?
I've been experiencing a persistent cough and shortness of breath for the past two weeks.
Have you had any fever or chest pain?
No, just the cough and difficulty breathing.
"""

transcript1 = """
हेलो doctor !
हेलो कैसे है आप ?

डॉक्टर बहुत तेज़ fever हैं.. उलटी भी हुई थी. Headache बहुत ज़ोर से है..

ठीक है.. let me check the temperature ! 104  फीवर है !

 I Would like to order a test ..

मलेरिया का टेस्ट करके देकते है.. आप आज सैंपल देदो.. कल रिजल्ट्स मिल जाएगा.. will see you tomorrow

तब तक आप Paracetemol ५०० mg और Vomikind १० mg दिन में ३ बार लेना..
"""

medical_info = extract_medical_info(transcript)
print(medical_info)